# 1. Импорт библиотек и настройки

In [47]:
import pandas as pd
pd.set_option('display.float_format', '{:.4f}'.format)
from pprint import pprint

# 2. Загрузка констант и динамики

In [48]:
const = {'Привлекаемые_средства': 380_000_000_000,
            'Ставка_купона_ОФЗ_ИН_л': 0.025,
            'Ставка_купона_ОФЗ_ПД': 0.1374,
            'Номинал_ОФЗ_ИН': 10_000,
            'Номинал_ОФЗ_ПД': 1000,
            'Количество_человек': 2_000_000,
            'НДФЛ': 0.13
        }
const

{'Привлекаемые_средства': 380000000000,
 'Ставка_купона_ОФЗ_ИН_л': 0.025,
 'Ставка_купона_ОФЗ_ПД': 0.1374,
 'Номинал_ОФЗ_ИН': 10000,
 'Номинал_ОФЗ_ПД': 1000,
 'Количество_человек': 2000000,
 'НДФЛ': 0.13}

In [49]:
# загружаем функии из модуля cbr_inflation
import function.cbr_inflation as cbr_inf
# загружаем из модуля new_function
from function.API_in_function import get_deposit_rates
from datetime import date

inf = cbr_inf.get_inflation()
# выбираем только последний год и последнее значение инфляции
inf_d = inf.copy()
inf_d['Год'] = inf_d['date'].dt.year
inf_d.rename(columns={
    'inflation': 'Инфляция',
    'target': 'Цель по инфляции'},inplace=True)
inf_d = inf_d[['Год','Инфляция']]
inf_d = inf_d.tail(1)
# автоматизируем продлжения ряда лет,и настраиваем вывод целовой инфляции
current_year = date.today().year
forecast_years = [ current_year + 1, current_year + 2]
inf2 = pd.DataFrame({
    'Год': forecast_years,
    'Инфляция': inf['target'].iloc[-1]})
inf_res = pd.concat([inf_d,inf2],ignore_index=True)

In [50]:
df = get_deposit_rates()
sd = df.tail(1)
# создаем переменную имеющую единственную последнюю актуальную ставку
value = sd['rate'].iloc[0]

In [51]:
DEPOSIT_DECREMENT = 2.5 # коэфициент снижения 
base = value
inf_res['Ставка депозита'] = base - (DEPOSIT_DECREMENT/100) * inf_res.index
inf_res[['Инфляция','Ставка депозита']] = inf_res[['Инфляция','Ставка депозита']]/100 # переводим проценты в числа
inf_res

,Год,Инфляция,Ставка депозита
0,2026,0.0602,0.1284
1,2027,0.0400,0.1281
2,2028,0.0400,0.1279


# 3. ОФЗ ИН (л)

In [52]:
ofz_in_l =inf_res.copy()
ofz_in_l

,Год,Инфляция,Ставка депозита
0,2026,0.0602,0.1284
1,2027,0.0400,0.1281
2,2028,0.0400,0.1279


In [53]:
ofz_in_l['Привлекаемые средства'] = const['Привлекаемые_средства']
ofz_in_l['Количество человек'] = const['Количество_человек']
ofz_in_l['Ставка купона'] = const['Ставка_купона_ОФЗ_ИН_л']
ofz_in_l

,Год,Инфляция,Ставка депозита,Привлекаемые средства,Количество человек,Ставка купона
0,2026,0.0602,0.1284,380000000000,2000000,0.0250
1,2027,0.0400,0.1281,380000000000,2000000,0.0250
2,2028,0.0400,0.1279,380000000000,2000000,0.0250


In [54]:
ofz_in_l['На руках у человека, руб'] = ofz_in_l['Привлекаемые средства'] / ofz_in_l['Количество человек']
ofz_in_l

,Год,Инфляция,Ставка депозита,Привлекаемые средства,Количество человек,Ставка купона,"На руках у человека, руб"
0,2026,0.0602,0.1284,380000000000,2000000,0.0250,190000.0000
1,2027,0.0400,0.1281,380000000000,2000000,0.0250,190000.0000
2,2028,0.0400,0.1279,380000000000,2000000,0.0250,190000.0000


In [55]:
ofz_in_l['Облигаций штук'] = ofz_in_l['На руках у человека, руб'] / const ['Номинал_ОФЗ_ИН']
ofz_in_l

,Год,Инфляция,Ставка депозита,Привлекаемые средства,Количество человек,Ставка купона,"На руках у человека, руб",Облигаций штук
0,2026,0.0602,0.1284,380000000000,2000000,0.0250,190000.0000,19.0000
1,2027,0.0400,0.1281,380000000000,2000000,0.0250,190000.0000,19.0000
2,2028,0.0400,0.1279,380000000000,2000000,0.0250,190000.0000,19.0000


In [56]:
ofz_in_l['Инфляционный множитель']= (1 + inf_res['Инфляция']).cumprod()
ofz_in_l

,Год,Инфляция,Ставка депозита,Привлекаемые средства,Количество человек,Ставка купона,"На руках у человека, руб",Облигаций штук,Инфляционный множитель
0,2026,0.0602,0.1284,380000000000,2000000,0.0250,190000.0000,19.0000,1.0602
1,2027,0.0400,0.1281,380000000000,2000000,0.0250,190000.0000,19.0000,1.1026
2,2028,0.0400,0.1279,380000000000,2000000,0.0250,190000.0000,19.0000,1.1467


In [57]:
ofz_in_l['Номинал после индексации'] = const['Номинал_ОФЗ_ИН']*ofz_in_l['Инфляционный множитель']
ofz_in_l

,Год,Инфляция,Ставка депозита,Привлекаемые средства,Количество человек,Ставка купона,"На руках у человека, руб",Облигаций штук,Инфляционный множитель,Номинал после индексации
0,2026,0.0602,0.1284,380000000000,2000000,0.0250,190000.0000,19.0000,1.0602,10602.0000
1,2027,0.0400,0.1281,380000000000,2000000,0.0250,190000.0000,19.0000,1.1026,11026.0800
2,2028,0.0400,0.1279,380000000000,2000000,0.0250,190000.0000,19.0000,1.1467,11467.1232


In [58]:
ofz_in_l['Номинал на начало'] = float(const['Номинал_ОФЗ_ИН'])
ofz_in_l.loc[ofz_in_l.index > 0, 'Номинал на начало'] = ofz_in_l['Номинал после индексации'].shift(1).fillna(const['Номинал_ОФЗ_ИН'])
ofz_in_l

,Год,Инфляция,Ставка депозита,Привлекаемые средства,Количество человек,Ставка купона,"На руках у человека, руб",Облигаций штук,Инфляционный множитель,Номинал после индексации,Номинал на начало
0,2026,0.0602,0.1284,380000000000,2000000,0.0250,190000.0000,19.0000,1.0602,10602.0000,10000.0000
1,2027,0.0400,0.1281,380000000000,2000000,0.0250,190000.0000,19.0000,1.1026,11026.0800,10602.0000
2,2028,0.0400,0.1279,380000000000,2000000,0.0250,190000.0000,19.0000,1.1467,11467.1232,11026.0800


In [59]:
ofz_in_l['Индексация номинала'] = ofz_in_l['Номинал на начало'] * ofz_in_l['Инфляция']
ofz_in_l

,Год,Инфляция,Ставка депозита,Привлекаемые средства,Количество человек,Ставка купона,"На руках у человека, руб",Облигаций штук,Инфляционный множитель,Номинал после индексации,Номинал на начало,Индексация номинала
0,2026,0.0602,0.1284,380000000000,2000000,0.0250,190000.0000,19.0000,1.0602,10602.0000,10000.0000,602.0000
1,2027,0.0400,0.1281,380000000000,2000000,0.0250,190000.0000,19.0000,1.1026,11026.0800,10602.0000,424.0800
2,2028,0.0400,0.1279,380000000000,2000000,0.0250,190000.0000,19.0000,1.1467,11467.1232,11026.0800,441.0432


In [60]:
ofz_in_l['Купон, руб'] = ofz_in_l['Номинал после индексации'] * ofz_in_l['Ставка купона']
ofz_in_l

,Год,Инфляция,Ставка депозита,Привлекаемые средства,Количество человек,Ставка купона,"На руках у человека, руб",Облигаций штук,Инфляционный множитель,Номинал после индексации,Номинал на начало,Индексация номинала,"Купон, руб"
0,2026,0.0602,0.1284,380000000000,2000000,0.0250,190000.0000,19.0000,1.0602,10602.0000,10000.0000,602.0000,265.0500
1,2027,0.0400,0.1281,380000000000,2000000,0.0250,190000.0000,19.0000,1.1026,11026.0800,10602.0000,424.0800,275.6520
2,2028,0.0400,0.1279,380000000000,2000000,0.0250,190000.0000,19.0000,1.1467,11467.1232,11026.0800,441.0432,286.6781


In [61]:
ofz_in_l['Доход без вычета'] = ofz_in_l['Купон, руб'] * ofz_in_l['Облигаций штук']
ofz_in_l

,Год,Инфляция,Ставка депозита,Привлекаемые средства,Количество человек,Ставка купона,"На руках у человека, руб",Облигаций штук,Инфляционный множитель,Номинал после индексации,Номинал на начало,Индексация номинала,"Купон, руб",Доход без вычета
0,2026,0.0602,0.1284,380000000000,2000000,0.0250,190000.0000,19.0000,1.0602,10602.0000,10000.0000,602.0000,265.0500,5035.9500
1,2027,0.0400,0.1281,380000000000,2000000,0.0250,190000.0000,19.0000,1.1026,11026.0800,10602.0000,424.0800,275.6520,5237.3880
2,2028,0.0400,0.1279,380000000000,2000000,0.0250,190000.0000,19.0000,1.1467,11467.1232,11026.0800,441.0432,286.6781,5446.8835


In [62]:
ofz_in_l ['Налоговый вычет, руб']= ofz_in_l['На руках у человека, руб'] * const['НДФЛ']
ofz_in_l

,Год,Инфляция,Ставка депозита,Привлекаемые средства,Количество человек,Ставка купона,"На руках у человека, руб",Облигаций штук,Инфляционный множитель,Номинал после индексации,Номинал на начало,Индексация номинала,"Купон, руб",Доход без вычета,"Налоговый вычет, руб"
0,2026,0.0602,0.1284,380000000000,2000000,0.0250,190000.0000,19.0000,1.0602,10602.0000,10000.0000,602.0000,265.0500,5035.9500,24700.0000
1,2027,0.0400,0.1281,380000000000,2000000,0.0250,190000.0000,19.0000,1.1026,11026.0800,10602.0000,424.0800,275.6520,5237.3880,24700.0000
2,2028,0.0400,0.1279,380000000000,2000000,0.0250,190000.0000,19.0000,1.1467,11467.1232,11026.0800,441.0432,286.6781,5446.8835,24700.0000


In [63]:
ofz_in_l['Доход с вычетом'] = ofz_in_l['Налоговый вычет, руб'] + ofz_in_l['Доход без вычета']
ofz_in_l

,Год,Инфляция,Ставка депозита,Привлекаемые средства,Количество человек,Ставка купона,"На руках у человека, руб",Облигаций штук,Инфляционный множитель,Номинал после индексации,Номинал на начало,Индексация номинала,"Купон, руб",Доход без вычета,"Налоговый вычет, руб",Доход с вычетом
0,2026,0.0602,0.1284,380000000000,2000000,0.0250,190000.0000,19.0000,1.0602,10602.0000,10000.0000,602.0000,265.0500,5035.9500,24700.0000,29735.9500
1,2027,0.0400,0.1281,380000000000,2000000,0.0250,190000.0000,19.0000,1.1026,11026.0800,10602.0000,424.0800,275.6520,5237.3880,24700.0000,29937.3880
2,2028,0.0400,0.1279,380000000000,2000000,0.0250,190000.0000,19.0000,1.1467,11467.1232,11026.0800,441.0432,286.6781,5446.8835,24700.0000,30146.8835


In [64]:
ofz_in_l = ofz_in_l[['Год', # перезаписываем в нужном порядке
 'Привлекаемые средства',
 'Количество человек',
 'Инфляция',
 'Ставка купона',
 'На руках у человека, руб',
 'Облигаций штук',
 'Инфляционный множитель',
 'Номинал на начало',
 'Индексация номинала',
 'Номинал после индексации',
 'Купон, руб',
 'Доход без вычета',
 'Налоговый вычет, руб',
 'Доход с вычетом']]


# 4. ОФЗ ПД

In [65]:
ofz_pd = ofz_in_l [['Год']].copy()
ofz_pd['Привлекаемые средства'] = const ["Привлекаемые_средства"]
ofz_pd ["Количество человек"] = const ["Количество_человек"]
ofz_pd ["Ставка купона"] = const ['Ставка_купона_ОФЗ_ПД']
ofz_pd ["На руках у человека"] = ofz_in_l [["На руках у человека, руб"]].copy()
ofz_pd ['Облигаций, штук'] = ofz_pd ['На руках у человека'] / const ['Номинал_ОФЗ_ПД']
ofz_pd ['Купон'] = const ['Номинал_ОФЗ_ПД'] * const ['Ставка_купона_ОФЗ_ПД']
ofz_pd ["Доход, руб"] = ofz_pd ["Купон"] * ofz_pd ['Облигаций, штук']
ofz_pd ['НДФЛ'] = ofz_pd ['Доход, руб'] * const ['НДФЛ']
ofz_pd ['Доход после вычета налога'] = ofz_pd['Доход, руб'] - ofz_pd ['НДФЛ']
ofz_pd

,Год,Привлекаемые средства,Количество человек,Ставка купона,На руках у человека,"Облигаций, штук",Купон,"Доход, руб",НДФЛ,Доход после вычета налога
0,2026,380000000000,2000000,0.1374,190000.0000,190.0000,137.4000,26106.0000,3393.7800,22712.2200
1,2027,380000000000,2000000,0.1374,190000.0000,190.0000,137.4000,26106.0000,3393.7800,22712.2200
2,2028,380000000000,2000000,0.1374,190000.0000,190.0000,137.4000,26106.0000,3393.7800,22712.2200


# 5. Депозит

In [66]:
depozit = ofz_in_l[['Год']].copy()
depozit ['Привлекаемые средства'] = const ['Привлекаемые_средства']
depozit ['Количество человек'] = const ['Количество_человек']
depozit ['На руках у человека'] = ofz_in_l ['На руках у человека, руб']
depozit ['Ставка депозита'] = inf_res['Ставка депозита'] 


In [67]:
depozit ['Коэфициент'] = (1 + inf_res['Ставка депозита'] / 12) **12
depozit

,Год,Привлекаемые средства,Количество человек,На руках у человека,Ставка депозита,Коэфициент
0,2026,380000000000,2000000,190000.0000,0.1284,1.1362
1,2027,380000000000,2000000,190000.0000,0.1281,1.1360
2,2028,380000000000,2000000,190000.0000,0.1279,1.1357


In [68]:
depozit ['Накопленный множитель'] = depozit ['Коэфициент'].cumprod()
depozit

,Год,Привлекаемые средства,Количество человек,На руках у человека,Ставка депозита,Коэфициент,Накопленный множитель
0,2026,380000000000,2000000,190000.0000,0.1284,1.1362,1.1362
1,2027,380000000000,2000000,190000.0000,0.1281,1.1360,1.2907
2,2028,380000000000,2000000,190000.0000,0.1279,1.1357,1.4658


In [69]:
initial_amount = depozit['На руках у человека'].iloc[0]

In [70]:
depozit ['Сумма на конец года'] = initial_amount * depozit ['Накопленный множитель']
depozit

,Год,Привлекаемые средства,Количество человек,На руках у человека,Ставка депозита,Коэфициент,Накопленный множитель,Сумма на конец года
0,2026,380000000000,2000000,190000.0000,0.1284,1.1362,1.1362,215884.1656
1,2027,380000000000,2000000,190000.0000,0.1281,1.1360,1.2907,245233.9269
2,2028,380000000000,2000000,190000.0000,0.1279,1.1357,1.4658,278504.9305


In [71]:
depozit ['Сумма на начало года'] = initial_amount
depozit.loc [depozit.index > 0, 'Сумма на начало года'] = depozit['Сумма на конец года']. shift (1)
# Более хорошая альтернатива:
# depozit['Сумма на начало года '] = depozit['Сумма на конец года'].shift(1).fillna(depozit['На руках у человека, руб'].iloc[0])
#depozit['Сумма начало'] = depozit['Сумма конец'].shift(1).fillna(initial_amount)
depozit

,Год,Привлекаемые средства,Количество человек,На руках у человека,Ставка депозита,Коэфициент,Накопленный множитель,Сумма на конец года,Сумма на начало года
0,2026,380000000000,2000000,190000.0000,0.1284,1.1362,1.1362,215884.1656,190000.0000
1,2027,380000000000,2000000,190000.0000,0.1281,1.1360,1.2907,245233.9269,215884.1656
2,2028,380000000000,2000000,190000.0000,0.1279,1.1357,1.4658,278504.9305,245233.9269


In [72]:
depozit['Проценты']= depozit['Сумма на конец года'] - depozit['Сумма на начало года']
depozit

,Год,Привлекаемые средства,Количество человек,На руках у человека,Ставка депозита,Коэфициент,Накопленный множитель,Сумма на конец года,Сумма на начало года,Проценты
0,2026,380000000000,2000000,190000.0000,0.1284,1.1362,1.1362,215884.1656,190000.0000,25884.1656
1,2027,380000000000,2000000,190000.0000,0.1281,1.1360,1.2907,245233.9269,215884.1656,29349.7613
2,2028,380000000000,2000000,190000.0000,0.1279,1.1357,1.4658,278504.9305,245233.9269,33271.0036


In [73]:
depozit = depozit [["Год", "Привлекаемые средства", # перезаписываем в нужном порядке
                   "Количество человек", 
                   "На руках у человека",
                   "Ставка депозита",
                   "Коэфициент",
                   "Накопленный множитель",
                   "Сумма на начало года",
                   "Сумма на конец года",
                    "Проценты"]]
depozit          

,Год,Привлекаемые средства,Количество человек,На руках у человека,Ставка депозита,Коэфициент,Накопленный множитель,Сумма на начало года,Сумма на конец года,Проценты
0,2026,380000000000,2000000,190000.0000,0.1284,1.1362,1.1362,190000.0000,215884.1656,25884.1656
1,2027,380000000000,2000000,190000.0000,0.1281,1.1360,1.2907,215884.1656,245233.9269,29349.7613
2,2028,380000000000,2000000,190000.0000,0.1279,1.1357,1.4658,245233.9269,278504.9305,33271.0036


# 6. Доход за период (собрал без merge)

In [74]:
ofz_in_l_summary = pd.DataFrame({
    'Инструмент': ['ОФЗ ИН (л)'],
    'На руках у человека': [ofz_in_l['На руках у человека, руб'].iloc[0]],
    'Доход, до налогов': [ofz_in_l ['Индексация номинала'].sum()*ofz_in_l['Облигаций штук'].iloc[0] +  ofz_in_l['Доход без вычета'].sum()],
})
ofz_in_l_summary ['Итоговая сумма'] = ofz_in_l['На руках у человека, руб'].iloc[0] + ofz_in_l_summary['Доход, до налогов'] + ofz_in_l ['Налоговый вычет, руб'].iloc [0] 
ofz_in_l_summary

,Инструмент,На руках у человека,"Доход, до налогов",Итоговая сумма
0,ОФЗ ИН (л),190000.0000,43595.5623,258295.5623


In [75]:
ofz_pd_summary = pd.DataFrame({
    'Инструмент': ['ОФЗ ПД'],
    'На руках у человека': [ofz_in_l ['На руках у человека, руб'].iloc[0]],
    'Доход, до налогов': [ofz_pd['Доход, руб'].sum()],
    'Итоговая сумма': [ofz_in_l ['На руках у человека, руб'].iloc[0] + ofz_pd['Доход, руб'].sum()]})
ofz_pd_summary

,Инструмент,На руках у человека,"Доход, до налогов",Итоговая сумма
0,ОФЗ ПД,190000.0000,78318.0000,268318.0000


In [76]:
depozit_summary = pd.DataFrame({
    'Инструмент': ['Депозит'],
    'На руках у человека': [ofz_in_l ['На руках у человека, руб'].iloc[0]],
    'Доход, до налогов': [depozit['Проценты'].sum()],
    'Итоговая сумма': [ofz_in_l ['На руках у человека, руб'].iloc[0] + depozit['Проценты'].sum()]})
depozit_summary

,Инструмент,На руках у человека,"Доход, до налогов",Итоговая сумма
0,Депозит,190000.0000,88504.9305,278504.9305


In [77]:
svodnay_dont_merge = pd.concat([ofz_in_l_summary, ofz_pd_summary,depozit_summary],ignore_index= True)
svodnay_dont_merge

,Инструмент,На руках у человека,"Доход, до налогов",Итоговая сумма
0,ОФЗ ИН (л),190000.0000,43595.5623,258295.5623
1,ОФЗ ПД,190000.0000,78318.0000,268318.0000
2,Депозит,190000.0000,88504.9305,278504.9305


In [78]:
inf_factor = (1 + inf_res['Инфляция']).prod()

In [79]:
svodnay_dont_merge['Очистка инфляции'] = svodnay_dont_merge['Итоговая сумма']/inf_factor
svodnay_dont_merge['Реальный доход'] = svodnay_dont_merge['Итоговая сумма'] - svodnay_dont_merge['На руках у человека']
svodnay_dont_merge

,Инструмент,На руках у человека,"Доход, до налогов",Итоговая сумма,Очистка инфляции,Реальный доход
0,ОФЗ ИН (л),190000.0000,43595.5623,258295.5623,225248.7898,68295.5623
1,ОФЗ ПД,190000.0000,78318.0000,268318.0000,233988.9398,78318.0000
2,Депозит,190000.0000,88504.9305,278504.9305,242872.5371,88504.9305


In [80]:
svodnay_dont_merge.loc[svodnay_dont_merge['Инструмент'] == 'ОФЗ ИН (л)', 'Очистка инфляции'] = None
# pohti_konec.loc[pohti_konec['Инструмент'] == 'ОФЗ ИН (л)', 'Реальный доход'] = None
svodnay_dont_merge

,Инструмент,На руках у человека,"Доход, до налогов",Итоговая сумма,Очистка инфляции,Реальный доход
0,ОФЗ ИН (л),190000.0000,43595.5623,258295.5623,NaN,68295.5623
1,ОФЗ ПД,190000.0000,78318.0000,268318.0000,233988.9398,78318.0000
2,Депозит,190000.0000,88504.9305,278504.9305,242872.5371,88504.9305


# Делаем таблицу с merge

In [81]:
df_s_merge = ofz_in_l[['Год', 'На руках у человека, руб']].copy()
df_s_merge.rename(columns={'На руках у человека, руб': 'Вложения'}, inplace=True)
df_s_merge

,Год,Вложения
0,2026,190000.0000
1,2027,190000.0000
2,2028,190000.0000


In [82]:
df_s_merge['ОФЗ ИН доход'] = ofz_in_l['Индексация номинала']* ofz_in_l['Облигаций штук'] + ofz_in_l['Доход без вычета']
df_s_merge

,Год,Вложения,ОФЗ ИН доход
0,2026,190000.0000,16473.9500
1,2027,190000.0000,13294.9080
2,2028,190000.0000,13826.7043


In [83]:
df_s_merge = df_s_merge.merge(ofz_pd[['Год', 'Доход, руб']], on='Год', how='left')
df_s_merge.rename(columns={'Доход, руб': 'ОФЗ ПД доход'}, inplace=True)
df_s_merge

,Год,Вложения,ОФЗ ИН доход,ОФЗ ПД доход
0,2026,190000.0000,16473.9500,26106.0000
1,2027,190000.0000,13294.9080,26106.0000
2,2028,190000.0000,13826.7043,26106.0000


In [84]:
df_s_merge = df_s_merge.merge(depozit[['Год', 'Проценты']], on='Год', how='left')
df_s_merge.rename(columns={'Проценты': 'Депозит доход'}, inplace=True)
df_s_merge

,Год,Вложения,ОФЗ ИН доход,ОФЗ ПД доход,Депозит доход
0,2026,190000.0000,16473.9500,26106.0000,25884.1656
1,2027,190000.0000,13294.9080,26106.0000,29349.7613
2,2028,190000.0000,13826.7043,26106.0000,33271.0036


# Делаем длинную 


In [85]:
# Теперь применяем melt к данным 
df_long = df_s_merge.melt(
    id_vars=['Год', 'Вложения'],
    value_vars=['ОФЗ ИН доход', 'ОФЗ ПД доход', 'Депозит доход'],
    var_name='Инструмент',
    value_name='Доход'
)

# Сортируем по году и инструменту
df_long = df_long.sort_values(['Год', 'Инструмент']).reset_index(drop=True)

df_long

,Год,Вложения,Инструмент,Доход
0,2026,190000.0000,Депозит доход,25884.1656
1,2026,190000.0000,ОФЗ ИН доход,16473.9500
2,2026,190000.0000,ОФЗ ПД доход,26106.0000
3,2027,190000.0000,Депозит доход,29349.7613
4,2027,190000.0000,ОФЗ ИН доход,13294.9080
5,2027,190000.0000,ОФЗ ПД доход,26106.0000
6,2028,190000.0000,Депозит доход,33271.0036
7,2028,190000.0000,ОФЗ ИН доход,13826.7043
8,2028,190000.0000,ОФЗ ПД доход,26106.0000


In [86]:
doxod_za_period = pd.DataFrame()
doxod_za_period =  df_long.groupby('Инструмент', as_index=False)['Доход'].sum()
#agg(Итоговая_сумма= ('Доход', sum) 

doxod_za_period

,Инструмент,Доход
0,Депозит доход,88504.9305
1,ОФЗ ИН доход,43595.5623
2,ОФЗ ПД доход,78318.0000


In [87]:
# Вычисляем итоговые суммы
total_oin = ofz_in_l['На руках у человека, руб'].iloc[0] + doxod_za_period.loc[doxod_za_period['Инструмент'] == 'ОФЗ ИН доход','Доход'].iloc[0] + ofz_in_l['Налоговый вычет, руб'].iloc[0]
total_pd = ofz_in_l['На руках у человека, руб'].iloc[0] + ofz_pd['Доход, руб'].sum()
total_dep = ofz_in_l['На руках у человека, руб'].iloc[0] + depozit['Проценты'].sum()

# Словарь соответствий
total_dict = {
    'ОФЗ ИН доход': total_oin,
    'ОФЗ ПД доход': total_pd,
    'Депозит доход': total_dep
}

# Добавляем колонку Итоговая сумма
doxod_za_period['Итоговая сумма'] = doxod_za_period['Инструмент'].map(total_dict)
doxod_za_period

,Инструмент,Доход,Итоговая сумма
0,Депозит доход,88504.9305,278504.9305
1,ОФЗ ИН доход,43595.5623,258295.5623
2,ОФЗ ПД доход,78318.0000,268318.0000


In [88]:
doxod_za_period['Очистка инфляции'] = doxod_za_period['Итоговая сумма']/inf_factor
doxod_za_period['Реальный доход'] = doxod_za_period['Итоговая сумма'] - ofz_in_l['На руках у человека, руб']
doxod_za_period.loc[doxod_za_period['Инструмент'] == 'ОФЗ ИН доход', 'Очистка инфляции'] = None
doxod_za_period

,Инструмент,Доход,Итоговая сумма,Очистка инфляции,Реальный доход
0,Депозит доход,88504.9305,278504.9305,242872.5371,88504.9305
1,ОФЗ ИН доход,43595.5623,258295.5623,NaN,68295.5623
2,ОФЗ ПД доход,78318.0000,268318.0000,233988.9398,78318.0000


# Нагрузка на государство


In [89]:
ofz_in_l_gos = inf_res.copy()
ofz_in_l_gos['Прибавка от инфляции'] = const['Привлекаемые_средства']*ofz_in_l_gos['Инфляция']
ofz_in_l_gos['Инфляционный множитель'] = (1+ofz_in_l_gos ['Инфляция']).cumprod()
ofz_in_l_gos['Тело долга'] = const['Привлекаемые_средства'] * ofz_in_l_gos['Инфляционный множитель']
ofz_in_l_gos['Расходы на купоны'] = ofz_in_l_gos['Тело долга'] * const['Ставка_купона_ОФЗ_ИН_л']
ofz_in_l_gos['Общие затраты'] = ofz_in_l_gos['Прибавка от инфляции']+ofz_in_l_gos['Расходы на купоны']
ofz_in_l_gos = ofz_in_l_gos[['Год','Инфляция','Прибавка от инфляции','Тело долга','Расходы на купоны','Общие затраты']]
ofz_in_l_gos

,Год,Инфляция,Прибавка от инфляции,Тело долга,Расходы на купоны,Общие затраты
0,2026,0.0602,22876000000.0000,402876000000.0000,10071900000.0000,32947900000.0000
1,2027,0.0400,15200000000.0000,418991040000.0000,10474776000.0000,25674776000.0000
2,2028,0.0400,15200000000.0000,435750681600.0000,10893767040.0000,26093767040.0000


In [90]:
cols = ['Прибавка от инфляции', 'Тело долга', 'Расходы на купоны','Общие затраты']
total_ofz_in_l_gos = pd.DataFrame(ofz_in_l_gos[cols].sum()).T
total_ofz_in_l_gos['Год'] = 'Итого'
itog_ofz_in_l_gos = pd.concat([ofz_in_l_gos,total_ofz_in_l_gos],ignore_index=True)
itog_ofz_in_l_gos

,Год,Инфляция,Прибавка от инфляции,Тело долга,Расходы на купоны,Общие затраты
0,2026,0.0602,22876000000.0000,402876000000.0000,10071900000.0000,32947900000.0000
1,2027,0.0400,15200000000.0000,418991040000.0000,10474776000.0000,25674776000.0000
2,2028,0.0400,15200000000.0000,435750681600.0000,10893767040.0000,26093767040.0000
3,Итого,NaN,53276000000.0000,1257617721600.0000,31440443040.0000,84716443040.0000


In [91]:
ofz_pd_gos = ofz_in_l_gos[['Год']].copy()
ofz_pd_gos['Тело долга'] = const['Привлекаемые_средства']
ofz_pd_gos['Расходы на купон'] = ofz_pd_gos['Тело долга'] * const['Ставка_купона_ОФЗ_ПД']
ofz_pd_gos['Сумма возврата,НДФЛ'] = ofz_pd['НДФЛ']*const['Количество_человек']
ofz_pd_gos['Итого при учете возврата НДФЛ'] = ofz_pd_gos['Расходы на купон']-ofz_pd_gos['Сумма возврата,НДФЛ']
ofz_pd_gos

,Год,Тело долга,Расходы на купон,"Сумма возврата,НДФЛ",Итого при учете возврата НДФЛ
0,2026,380000000000,52212000000.0000,6787560000.0000,45424440000.0000
1,2027,380000000000,52212000000.0000,6787560000.0000,45424440000.0000
2,2028,380000000000,52212000000.0000,6787560000.0000,45424440000.0000


In [92]:
cols2 = ['Расходы на купон','Сумма возврата,НДФЛ','Итого при учете возврата НДФЛ']
total_ofz_pd_gos = pd.DataFrame([ofz_pd_gos[cols2].sum()])
total_ofz_pd_gos['Год'] = 'Итого'
itog_ofz_pd_gos = pd.concat([ofz_pd_gos,total_ofz_pd_gos],ignore_index=True)
itog_ofz_pd_gos

,Год,Тело долга,Расходы на купон,"Сумма возврата,НДФЛ",Итого при учете возврата НДФЛ
0,2026,380000000000.0000,52212000000.0000,6787560000.0000,45424440000.0000
1,2027,380000000000.0000,52212000000.0000,6787560000.0000,45424440000.0000
2,2028,380000000000.0000,52212000000.0000,6787560000.0000,45424440000.0000
3,Итого,NaN,156636000000.0000,20362680000.0000,136273320000.0000
